# dcc_11

データ収集とクリーニング

11-データ加工：ビニング、エンコーディング

---

## 【要点まとめ】 データ加工：ビニング、エンコーディング

### **データ加工（ビニング、エンコーディング）**

#### **機械学習モデルとデータ形式の「壁」**
多くの機械学習アルゴリズム（回帰、分類など）は、その内部で数学的な計算（距離の計算や行列演算）を行います。そのため、**入力として受け付けることができるのは原則として「数値」のみ**です。

しかし、私たちが扱う現実のデータは「数値」だけではありません。

1. 数値だが、そのまま使うと問題があるデータ（例: 年齢、価格）
2. そもそも数値ではないデータ（例: カテゴリ、住所）

これらのデータを、モデルが計算できるように適切に変換する処理が「ビニング」と「エンコーディング」です。

#### **ビニング (Binning): 連続値をカテゴリ値へ**

**ビニング**（または離散化）とは、``年齢``（18, 19, 20, ...）や``価格``（1000, 1001, 1002, ...）のような連続的な数値データを、「10代」「20代」や「低価格帯」「中価格帯」といった **カテゴリ的な区間（ビン）** に変換する処理です。

#### **なぜビニングが必要か？**

- **過学習（Overfitting）の防止**:もし価格データに``1000円``と``1001円``という値があった場合、機械学習モデル（特に決定木など）は、この「1円」の違いに何か重要な意味があると誤って学習し、データに対して過剰に適合してしまう（過学習）可能性があります。ビニングによって「0〜5000円は低価格帯」と丸めることで、このような細かすぎるノイズの影響を減らし、モデルの汎化性能を高めることができます。

- **非線形性の単純化**:「20代は購買意欲が高いが、30代になると一度下がり、50代でまた上がる」といった複雑な関係（非線形性）を、モデルがそのまま学習するのは難しい場合があります。ビニングによって「年代」というカテゴリに変換することで、モデルは「20代のグループ」「30代のグループ」といった単位で特徴を捉えやすくなります。

In [ ]:
# ビニングのコード例
import pandas as pd
df_bin = pd.DataFrame({'価格': [500, 1200, 6000, 25000]})

# 1. ビン（区間）の境界値を定義
bins = [0, 5000, 20000, float('inf')] # 0～5000, 5000～20000, 20000～無限大

# 2. 各ビンのラベル（名前）を定義
labels = ['低価格帯', '中価格帯', '高価格帯']

# 3. pd.cut を使ってビニング
df_bin['価格帯'] = pd.cut(df_bin['価格'], bins=bins, labels=labels, right=True)
df_bin

#### **エンコーディング (Encoding): 文字列を数値へ**

**エンコーディング**とは、``Category``列の「Electronics」や「Books」といったカテゴリカルデータ（文字列）を、モデルが計算できる数値に変換する処理です。

#### **手法1: ラベルエンコーディング (Label Encoding)**

単純に、各カテゴリ名を一意の整数に置き換える方法です。

- **問題点**: モデルが、これらの数値間に**本来存在しない順序関係（大小関係）**があると誤解する可能性があります。（例: Toys(2) > Electronics(1) のような数学的な関係を学習してしまう）

- **適したケース**: 「小」「中」「大」のように、元々順序関係があるカテゴリ（順序変数）には適しています。

In [ ]:
# ラベルエンコーディングのコード例
from sklearn.preprocessing import LabelEncoder
df_enc = pd.DataFrame({'カテゴリ': ['Books', 'Electronics', 'Books', 'Toys']})

# 1. LabelEncoderのインスタンスを作成
le = LabelEncoder()

# 2. .fit_transform() で学習と変換を同時に行う
df_enc['カテゴリ_Label'] = le.fit_transform(df_enc['カテゴリ'])
df_enc

#### **手法2: ワンホットエンコーディング (One-Hot Encoding)**

カテゴリ間に順序関係がないことをモデルに正しく伝えるための、最も一般的な手法です。各カテゴリを新しい独立した列（ダミー変数）とし、該当するカテゴリの列だけを``1``、それ以外を``0``で表現します。

- **長所**: カテゴリ同士が対等な関係であることをモデルに正しく示すことができます。順序の誤解を防ぐため、機械学習では原則としてこちらを使用します。

- **短所**: カテゴリの種類が非常に多い（例: ``市区町村``が1000以上ある）場合、列の数が爆発的に増加してしまいます（次元の呪い）。



In [ ]:
# ワンホットエンコーディングのコード例
df_enc = pd.DataFrame({'カテゴリ': ['Books', 'Electronics', 'Books', 'Toys']})

# pd.get_dummies() を使うのが最も簡単
df_ohe = pd.get_dummies(df_enc, columns=['カテゴリ'])
df_ohe

### **まとめ**

**実践的応用**:
- **ビニング**は、顧客の「年代」別分析や、商品の「価格帯」別分析など、マーケティング分析で頻繁に使用されます。
- **エンコーディング**は、機械学習モデルにカテゴリデータ（例: 天気、性別、商品カテゴリ）を入力するための必須の前処理です。

**ポイント**:
- 機械学習モデルは数値しか計算できない、という大原則を理解することが重要です。
- **ビニング**は、数値データのノイズを減らし、モデルが傾向を掴みやすくするために行います。
- **エンコーディング**は、モデルが計算できるように文字列データを数値に変換するために必須の処理です。
- **順序関係のないカテゴリ**（例: ``Books``, ``Electronics``）には**ワンホットエンコーディング**を、**順序関係のあるカテゴリ**（例: ``低``, ``中``, ``高``）には**ラベルエンコーディング**を選択するという使い分けが、モデルの精度を左右する重要なポイントとなります。

## 11-課題
- dcc_11: エンコーディング手法の比較

### dcc_11
``pandas``の ``read_csv`` 関数を使い、``data_11``内のファイルをDataFrameに読み込んでください。「物件の立地（``Location``）」というカテゴリカルデータだけを使って、「家賃（``Price``）」という数値を予測する回帰モデルを構築してみます。カテゴリカルデータを数値に変換する2つの主要な手法、「ラベルエンコーディング」と「ワンホットエンコーディング」が、機械学習モデルの予測性能にどのような影響を与えるかを比較・考察したエンコーディング手法考察レポートを完成させてください。

In [ ]:
# --- ライブラリのインポート ---
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import numpy as np

# --- データの読み込み ---
filepath = 'data_11/data_11.csv'
df = pd.read_csv(filepath)

# 目的変数y（予測したいもの）: 家賃
y = df['Price']

print("--- 元のデータ (先頭5行) ---")
display(df.head())

In [ ]:
# --- ラベルエンコーディング --- 
le = LabelEncoder()
X_label = le.fit_transform(df['Location'])

# A駅=0, B駅=1, C駅=2 (アルファベット順) と変換されたことを確認
print("--- ラベルエンコーディングのマッピング ---")
print(f"カテゴリ: {list(le.classes_)}")
print(f"変換後の数値: {le.transform(le.classes_)}")

# .reshape(-1, 1) は、1次元の配列をモデルが要求する2次元の形状に変換しています
X_label = X_label.reshape(-1, 1)

print("\n--- ラベルエンコーディング後のX (先頭5行) ---")
print(X_label[:5])

In [ ]:
# --- モデル1の学習と評価 ---
model1 = LinearRegression()
model1.fit(X_label, y)
y_pred1 = model1.predict(X_label)
r2_label = r2_score(y, y_pred1)

print(f"\nモデル1 (ラベルエンコーディング) のR2スコア: {r2_label:.3f}")

In [ ]:
# --- ワンホットエンコーディング --- 
# drop_first=True は、3つのカテゴリのうち2つが0なら残りは1だと自動的にわかるため、列を1つ減らす（多重共線性）対策です。
X_ohe = pd.get_dummies(df['Location'], drop_first=True)

print("--- ワンホットエンコーディング後のX (先頭5行) ---")
display(X_ohe.head())

In [ ]:
# --- モデル2の学習と評価 ---
model2 = LinearRegression()
model2.fit(X_ohe, y)
y_pred2 = model2.predict(X_ohe)
r2_ohe = r2_score(y, y_pred2)

print(f"\nモデル2 (ワンホットエンコーディング) のR2スコア: {r2_ohe:.3f}")

#### **エンコーディング手法考察レポート**
「物件の立地（``Location``）」というカテゴリカルデータだけを使って、「家賃（``Price``）」という数値を予測する回帰モデルを構築してみます。カテゴリカルデータを数値に変換する2つの主要な手法、「ラベルエンコーディング」と「ワンホットエンコーディング」が、機械学習モデルの予測性能にどのような影響を与えるかを比較した。
- ラベルエンコーディングを使用した場合、モデルの予測性能（決定係数 R2スコア）は ____ であった。
- ワンホットエンコーディング を使用した場合、モデルの予測性能（決定係数 R2スコア）は ____ であった。

この結果から、今回のケースでは ____ を用いた方が、より高い予測性能を発揮できることがわかった。
これは、Location（立地）が本来は順序関係を持たないカテゴリであるにもかかわらず、ラベルエンコーディングが作成した数値（A駅=0, B駅=1, C駅=2）に、モデルが「____」があると誤って学習してしまったためと考えられる。